# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. The dataset covers ordered logistic regression outputs, socio-demographic characteristics, and adoption behaviors related to rangeland management practices in Northern Kenya.

### Dataset Source
The dataset is provided via its Croissant schema:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading

Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"License: {meta.license}")
print(f"Citation: {getattr(meta, 'cite_as', getattr(meta, 'citeAs', None))}")

## 2. Data Overview

We inspect the record sets (tables) and available fields. All entities are referenced using their `@id` values as defined in the Croissant schema.

In [ ]:
# List available record sets and their @id
print("Available record sets (by @id):")
record_sets = []
for rs in dataset.record_sets:
    print(f" - {rs['@id']} | Name: {rs.get('name', '')}")
    record_sets.append(rs['@id'])

Let's explore the structure of each record set: the fields (columns), their `@id`s and data types.

In [ ]:
# Preview fields in each record set by @id
for rs in dataset.record_sets:
    print(f"\nRecord set: {rs['@id']}")
    if 'field' in rs:
        for field in rs['field']:
            print(f" - Field @id: {field['@id']}, Name: {field.get('name')}, Data type: {field.get('dataType')}")

To view actual records, we can iterate over any record set and print a few results. (Replace `<record_set_id>` with one from above as needed.)

In [ ]:
# Example: Previewing records in a record set
# (Use an actual @id from above. If multiple record sets exist, replace the value below accordingly.)

if len(record_sets) > 0:
    preview_set_id = record_sets[0]
    print(f"\nShowing sample records from: {preview_set_id}\n")
    for i, row in enumerate(dataset.records(record_set=preview_set_id)):
        print(row)
        if i >= 2:
            break

## 3. Data Extraction

We load all tabular record sets into Pandas DataFrames. All record set and field references use their `@id` as required for reproducibility.

In [ ]:
dfs = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f"Loaded DataFrame for {rs_id} with shape {df.shape}")
    else:
        print(f"No records found for {rs_id}")

# Pick a main record set for EDA (replace with real @id if desired)
if dfs:
    main_record_set_id = list(dfs.keys())[0]
    print(f"\nFields in record set {main_record_set_id}:")
    print(dfs[main_record_set_id].columns.tolist())
    dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Here, we demonstrate common EDA tasks, such as filtering rows, normalizing a numeric field, and grouping by a categorical attribute. (Update field `@id`s as needed to reflect the schema in use.)

In [ ]:
if dfs:
    # Inspect DataFrame columns again for reference
    df = dfs[main_record_set_id]
    print(f"Available columns: {df.columns.tolist()}")

    # Example field @id selection (update as needed):
    # Let's try to automatically select a numeric column
    numeric_field = None
    for col in df.columns:
        # Test if column can be converted to float
        try:
            pd.to_numeric(df[col].dropna().head(20))
            numeric_field = col
            break
        except Exception:
            continue

    if numeric_field is None:
        print("No numeric field detected for demonstration.")
    else:
        print(f"Using numeric field: {numeric_field}")
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping (pick first non-numeric field as group key)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by: {group_field}")
            grouped_df = (
                filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            )
            print(grouped_df.head())

## 5. Visualization

We visualize the distribution of the selected numeric field and, if available, its relationship with a grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load a Croissant-structured dataset with `mlcroissant`
- List and access record sets and fields using their `@id`s
- Extract records into `pandas` DataFrames
- Perform basic filtering, normalization, grouping, and visualize fields

This process ensures reproducibility and clarity of data exploration for the FAIR² dataset and enables further statistical or machine learning analyses. For deeper analysis, consult the dataset documentation for the meaning of each `@id` and variable.